In [1]:
import pandas as pd

In [2]:
pm10 = pd.read_excel("PM10_Letna_Povprecja_Po_Postajah.xlsx")
pm25 = pd.read_excel("PM25_Letna_Povprecja_Po_Postajah.xlsx")
rak = pd.read_csv("podatki_pljucni_rak.csv")

In [3]:
pm10 = pm10.rename(columns={
    "Leto": "Leto",
    "Postaja": "Postaja",
    "Povprečje PM10 (µg/m³)": "Povprečje PM10 (µg/m³)"
})

pm25 = pm25.rename(columns={
    "Leto": "Leto",
    "Postaja": "Postaja",
    "Povprečje PM2.5 (µg/m³)": "Povprečje PM2.5 (µg/m³)"
})

rak = rak.rename(columns={
    "Leto": "Leto",
    "Občina": "Občina",
    "Novi primeri pljučnega raka": "Novi primeri pljučnega raka",
    "Umrljivost zaradi pljučnega raka (0–74 let)": "Umrljivost zaradi pljučnega raka (0–74 let)"
})

In [4]:
def popravi_znake(naziv):
    if pd.isna(naziv):
        return naziv
    return (naziv
        .replace("Å¾", "ž")
        .replace("Å¡", "š")
        .replace("Å ", "Š")
        .replace("Å½", "Ž")
        .replace("Ä", "č")
        .replace("Ä", "Đ")
        .strip())

pm10["Postaja"] = pm10["Postaja"].apply(popravi_znake)
pm25["Postaja"] = pm25["Postaja"].apply(popravi_znake)

In [5]:
mapping = {
    # Ljubljana
    "Ljubljana-Bežigrad": "Ljubljana",
    "Ljubljana-BF": "Ljubljana",
    "Ljubljana-Vič": "Ljubljana",
    "Ljubljana-Celovška": "Ljubljana",
    "Ljubljana-prometna": "Ljubljana",

    # Maribor
    "Maribor-Vrbanski": "Maribor",
    "Maribor-Titova": "Maribor",
    "Maribor-center": "Maribor",

    # Celje
    "Celje-Bolnica": "Celje",
    "Celje-Ljubljanska": "Celje",
    "Celje-Mariborska": "Celje",

    # Nova Gorica
    "Nova Gorica-Vojkova": "Nova Gorica",
    "Nova Gorica-Grčna": "Nova Gorica",

    # Murska Sobota
    "Murska Sobota-Cankarjeva": "Murska Sobota",
    "Murska Sobota-Rakičan": "Murska Sobota",

    # Druga večja mesta
    "Kranj": "Kranj",
    "Koper": "Koper",
    "Novo mesto": "Novo mesto",
    "Velenje": "Velenje",
    "Zagorje": "Zagorje ob Savi",
    "Hrastnik": "Hrastnik",
    "Trbovlje": "Trbovlje",
    "Iskrba": "Kočevje",
    "Ilirska Bistrica-Gregorčičeva": "Ilirska Bistrica",
    "Ilirska Bistrica-Rečica": "Ilirska Bistrica"
}

pm10["Občina"] = pm10["Postaja"].map(mapping)
pm25["Občina"] = pm25["Postaja"].map(mapping)

In [6]:
pm10_obcine = (
    pm10.groupby(["Leto", "Občina"], as_index=False)["Povprečje PM10 (µg/m³)"]
    .mean()
)
pm25_obcine = (
    pm25.groupby(["Leto", "Občina"], as_index=False)["Povprečje PM2.5 (µg/m³)"]
    .mean()
)


In [7]:
zrak = pd.merge(pm10_obcine, pm25_obcine, on=["Leto", "Občina"], how="outer")

koncni = pd.merge(rak, zrak, on=["Leto", "Občina"], how="left")

In [8]:
koncni.to_csv("Združeni_Podatki_PM_Rak_Po_Občinah.csv", index=False, encoding="utf-8-sig")

print("Končani podatki so shranjeni.")
print("Število vrstic:", len(koncni))
koncni

Končani podatki so shranjeni.
Število vrstic: 2118


,Leto,Občina,Novi primeri pljučnega raka,Umrljivost zaradi pljučnega raka (0–74 let),Povprečje PM10 (µg/m³),Povprečje PM2.5 (µg/m³)
0,2016,Ajdovščina,NaN,30.5405329658894,NaN,NaN
1,2016,Apače,NaN,71.5752476382795,NaN,NaN
2,2016,Beltinci,NaN,61.5799774480047,NaN,NaN
3,2016,Benedikt,NaN,37.418345720247,NaN,NaN
4,2016,Bistrica ob Sotli,NaN,26.9462813594685,NaN,NaN
...,...,...,...,...,...,...
2113,2025,Železniki,79.495,55.537,NaN,NaN
2114,2025,Žetale,64.328,40.784,NaN,NaN
2115,2025,Žiri,48.558,22.77,NaN,NaN
2116,2025,Žirovnica,53.24,21.595,NaN,NaN
